# Toxicity Detection — End-to-End Notebook

Covers the full lifecycle in one place:
1. Data loading & stratified splitting
2. Tokenization (no leakage)
3. Training with MLflow
4. Evaluation (accuracy, F1, ROC-AUC, confusion matrix)
5. Bias analysis
6. Inference latency benchmark
7. Model card generation

In [ ]:
# Install dependencies (if not already installed)
# !pip install -r requirements.txt

In [ ]:
import json, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
warnings.filterwarnings('ignore')

# Load config
with open('config.yaml') as f:
    cfg = yaml.safe_load(f)
print('Config loaded:', list(cfg.keys()))

## 1. Data Loading & Stratified Split

In [ ]:
from train_toxicity_model import load_and_split, fix_seed

fix_seed(cfg['training']['seed'])
datasets = load_and_split(cfg, smoke_test=True)  # smoke_test=False for full run

print('Train:', len(datasets['train']), 'rows')
print('Val  :', len(datasets['validation']), 'rows')
print('Test :', len(datasets['test']), 'rows')

# Verify stratification
for split_name in ('train', 'validation', 'test'):
    labels = datasets[split_name]['label']
    pos_rate = sum(labels) / len(labels)
    print(f'  {split_name}: positive_rate = {pos_rate:.3f}')

In [ ]:
# Class distribution plot
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, split_name in zip(axes, ('train', 'validation', 'test')):
    labels = datasets[split_name]['label']
    counts = pd.Series(labels).value_counts()
    ax.bar(['non-toxic', 'toxic'], [counts.get(0, 0), counts.get(1, 0)],
           color=['#2ecc71', '#e74c3c'])
    ax.set_title(split_name)
    ax.set_ylabel('Count')
plt.suptitle('Class Distribution (stratified)', fontweight='bold')
plt.tight_layout()
plt.savefig('artifacts/class_distribution.png', dpi=120)
plt.show()

## 2. Tokenization

**Key anti-leakage property:** We use the pre-trained tokenizer's fixed vocabulary.
No fitting on our data — zero leakage risk.

In [ ]:
from transformers import AutoTokenizer
from train_toxicity_model import tokenize_datasets

tokenizer = AutoTokenizer.from_pretrained(cfg['model']['pretrained_name'])
tokenized = tokenize_datasets(datasets, tokenizer, cfg['model']['max_length'])

# Inspect a sample
sample = tokenized['train'][0]
print('Keys:', list(sample.keys()))
print('Input IDs (first 10):', sample['input_ids'][:10])
print('Label:', sample['label'])

## 3. Training with MLflow

In [ ]:
from train_toxicity_model import train

# Use smoke_test=False for a real training run
results = train(cfg, smoke_test=True)
print('\nMLflow run_id:', results['run_id'])

## 4. Evaluation Metrics

In [ ]:
# Load saved evaluation results
import os
results_path = 'artifacts/toxicity_model/evaluation_results.json'
if os.path.exists(results_path):
    with open(results_path) as f:
        eval_results = json.load(f)
    
    test = eval_results['test']
    metrics_df = pd.DataFrame({
        'Split': ['Validation', 'Test'],
        'Accuracy': [eval_results['validation']['accuracy'], test['accuracy']],
        'F1': [eval_results['validation']['f1'], test['f1']],
        'Precision': [eval_results['validation']['precision'], test['precision']],
        'Recall': [eval_results['validation']['recall'], test['recall']],
        'ROC-AUC': [eval_results['validation']['roc_auc'], test['roc_auc']],
    })
    print(metrics_df.to_string(index=False, float_format='{:.4f}'.format))

In [ ]:
# Confusion Matrix
if os.path.exists(results_path):
    cm = np.array(eval_results['test']['confusion_matrix'])
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Predicted Non-Toxic', 'Predicted Toxic'],
                yticklabels=['Actual Non-Toxic', 'Actual Toxic'], ax=ax)
    ax.set_title('Confusion Matrix — Test Set', fontweight='bold')
    plt.tight_layout()
    plt.savefig('artifacts/confusion_matrix.png', dpi=120)
    plt.show()

## 5. Bias Analysis

In [ ]:
from evaluate_bias import evaluate_bias

bias_report = evaluate_bias(cfg, model_dir='artifacts/toxicity_model')
print('Fairness check passed:', bias_report['passed_fairness'])
if bias_report['violations']:
    print('Violations:')
    for v in bias_report['violations']:
        print(' ', v)

In [ ]:
# Subgroup F1 gap chart
subgroups = bias_report.get('subgroups', {})
if subgroups:
    sg_df = pd.DataFrame(subgroups).T[['n', 'f1', 'gap_f1', 'fpr']].reset_index()
    sg_df.columns = ['group', 'n', 'f1', 'gap_f1', 'fpr']
    sg_df = sg_df.sort_values('gap_f1')

    fig, ax = plt.subplots(figsize=(10, max(4, len(sg_df) * 0.4)))
    colors = ['#e74c3c' if abs(g) > bias_report['fairness_threshold'] else '#3498db'
              for g in sg_df['gap_f1']]
    ax.barh(sg_df['group'], sg_df['gap_f1'], color=colors)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.axvline(bias_report['fairness_threshold'], color='red', linestyle='--', alpha=0.5, label='Threshold')
    ax.axvline(-bias_report['fairness_threshold'], color='red', linestyle='--', alpha=0.5)
    ax.set_xlabel('F1 Gap vs Overall Baseline')
    ax.set_title('Subgroup F1 Gap (red = fairness violation)', fontweight='bold')
    ax.legend()
    plt.tight_layout()
    plt.savefig('artifacts/bias_f1_gap.png', dpi=120)
    plt.show()

## 6. Inference Latency Benchmark

In [ ]:
from inference import ToxicityClassifier

clf = ToxicityClassifier(
    model_dir='artifacts/toxicity_model',
    sla_ms=cfg['inference']['sla_ms'],
)

# Single sample test
r = clf.predict('You are a terrible human being!')
print(json.dumps(r, indent=2))

In [ ]:
# Latency benchmark
bench = clf.benchmark(
    warmup_runs=cfg['inference']['warmup_runs'],
    benchmark_runs=cfg['inference']['benchmark_runs'],
)
print(json.dumps(bench, indent=2))

# Save for model card
import os
os.makedirs('artifacts', exist_ok=True)
with open('artifacts/latency_report.json', 'w') as f:
    json.dump(bench, f, indent=2)

## 7. Model Card Generation

In [ ]:
from generate_model_card import generate_model_card, _load_json

card = generate_model_card(
    cfg=cfg,
    eval_results=_load_json('artifacts/toxicity_model/evaluation_results.json'),
    bias_report=_load_json('artifacts/bias_report.json'),
    latency_report=_load_json('artifacts/latency_report.json'),
)

with open('model_card.yaml', 'w') as f:
    yaml.dump(card, f, default_flow_style=False, sort_keys=False)

print('Model card written to model_card.yaml')
print('\n=== Quick Summary ===')
print(f"Model   : {card['model_details']['name']}")
print(f"Date    : {card['model_details']['date']}")
print(f"Test F1 : {card['evaluation_metrics']['test_set']['f1']}")
print(f"Fairness: {'PASS' if card['bias_and_fairness']['passed_fairness_check'] else 'FAIL'}")
print(f"SLA     : {'PASS' if card['inference_performance']['latency_benchmark'].get('sla_passed') else 'PENDING'}")